In [ ]:
# ============================================================
# LOG 
# ============================================================

import logging

# Remove handlers existentes do Colab e reconfigura
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)

logger = logging.getLogger("datathon.churn")
logger.info("Logger configurado corretamente ✅")


In [ ]:
# ============================================================
# Imports e Configuração de Reprodutibilidade
# ============================================================

import pandas as pd
import numpy as np
import os
import random

# --- Reprodutibilidade ---
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

logger.info("Seeds fixadas — SEED=%d", SEED)

# se dois membros do time rodarem o mesmo código, devem obter exatamente os mesmos resultados.
# O PYTHONHASHSEED controla aleatoriedade do próprio Python (dicionários, sets),
# o random.seed controla a lib padrão, e o np.random.seed controla o numpy.

In [ ]:
# ============================================================
# Montar Google Drive e carregar CSV
# ============================================================

from google.colab import drive

drive.mount("/content/drive")
logger.info("Google Drive montado com sucesso")

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/fase 5/data/raw/Customer-Churn-Records.csv"

df_raw = pd.read_csv(DATA_PATH)

# Colunas que não são features (identificadores e leakage)
# RowNumber	Identificador	É apenas um índice sequencial; pode causar leakage pela ordem dos dados.
# CustomerId	Identificador	Valor único por cliente; não carrega padrão comportamental.
# Surname	Identificador	Alta cardinalidade; o nome de alguém não define sua saúde financeira.
COLS_TO_DROP = ["RowNumber", "CustomerId", "Surname"]
TARGET_COL   = "Exited"

df = df_raw.drop(columns=COLS_TO_DROP)

logger.info("Arquivo carregado: %s", DATA_PATH)
logger.info("Shape df_raw: %s", df_raw.shape)
logger.info("df -> Shape: %s", df.shape)
logger.info("Colunas finais: %s", df.columns.tolist())
logger.info("Distribuição do target (Churn):\n%s", df[TARGET_COL].value_counts().to_string())

In [ ]:
df.head()

## 📋 Dicionário de Dados — Bank Customer Churn

| Coluna | Tradução (PT-BR) | Tipo | Restrições | Explicação |
|---|---|---|---|---|
| CreditScore | Pontuação de Crédito | int | 300–850 | Confiabilidade financeira do cliente |
| Geography | País | str | France, Germany, Spain | País de residência |
| Gender | Gênero | str | Male, Female | Gênero do cliente |
| Age | Idade | int | 18–100 | Idade em anos |
| Tenure | Tempo de Casa | int | 0–10 | Anos como cliente do banco |
| Balance | Saldo | float | ≥ 0 | Valor disponível em conta |
| NumOfProducts | Nº de Produtos | int | 1–4 | Serviços bancários ativos |
| HasCrCard | Possui Cartão | int | 0 ou 1 | 1 = Sim |
| IsActiveMember | Membro Ativo | int | 0 ou 1 | 1 = Movimenta a conta |
| EstimatedSalary | Salário Estimado | float | > 0 | Rendimento anual estimado |
| **Exited** | **Churn (Target)** | int | 0 ou 1 | **1 = Saiu, 0 = Ficou** |
| Complain | Reclamação | int | 0 ou 1 | 1 = Registrou reclamação |
| Satisfaction Score | Score de Satisfação | int | 1–5 | Nota dada pelo cliente |
| Card Type | Tipo de Cartão | str | DIAMOND, GOLD, SILVER, PLATINUM | Categoria do cartão |
| Point Earned | Pontos de Fidelidade | int | ≥ 0 | Pontos acumulados |

In [ ]:
# ============================================================
# Feature Engineering
#
# Etapas:
#   1. Definição de escopo (prevenção de data leakage)
#   2. Tratamento de variáveis categóricas
#   3. Criação de novas features
#   4. Seleção final e escalonamento
# ============================================================

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df_feat = df.copy()

# --- 1. Definição de Escopo (Prevenção de Data Leakage) ---
# Variáveis que só existem após o churn não podem ser usadas no treino:
#   - 'Complain'          → ocorre DEPOIS que o cliente decidiu sair (leakage)
#   - 'Satisfaction Score'→ ocorre DEPOIS que o cliente decidiu sair (leakage)
#   - 'Exited'            → é o nosso target
COLS_LEAKAGE = [TARGET_COL, "Complain", "Satisfaction Score"]
TARGET_COL   = "Exited"

logger.info("Target: %s | Colunas removidas por leakage: %s",
            TARGET_COL, COLS_LEAKAGE)

# --- 2. Tratamento de Variáveis Categóricas ---
# A. Ordinal Encoding para 'Card Type'
#    Existe hierarquia lógica: SILVER < GOLD < PLATINUM < DIAMOND
#    Usar LabelEncoder aqui seria arbitrário e introduziria ruído
card_mapping = {"SILVER": 1, "GOLD": 2, "PLATINUM": 3, "DIAMOND": 4}
df_feat["Card Type"] = df_feat["Card Type"].map(card_mapping)
logger.info("Card Type — Ordinal Encoding aplicado: %s", card_mapping)

# B. One-Hot Encoding para 'Geography'
#    Países não têm hierarquia — criar uma seria introduzir viés
#    drop_first=True evita a Dummy Variable Trap (multicolinearidade)
#    Resultado: colunas Geo_Germany e Geo_Spain (France é a referência)
df_feat = pd.get_dummies(df_feat, columns=["Geography"],
                         prefix="Geo", drop_first=True)
geo_cols = [c for c in df_feat.columns if c.startswith("Geo_")]
logger.info("Geography — One-Hot Encoding aplicado: %s", geo_cols)

# C. Label Encoding para 'Gender'
#    Binário (Male/Female) — Label Encoding é suficiente
le_gender = LabelEncoder()
df_feat["Gender"] = le_gender.fit_transform(df_feat["Gender"])
logger.info("Gender — Label Encoding: %s",
            dict(zip(le_gender.classes_,
                     le_gender.transform(le_gender.classes_))))

# --- 3. Criação de Novas Features ---
# BalancePerProduct: concentração de patrimônio no banco
# replace(0,1) evita divisão por zero se NumOfProducts for 0
df_feat["BalancePerProduct"] = (
    df_feat["Balance"] / df_feat["NumOfProducts"].replace(0, 1)
)
logger.info("Feature criada: BalancePerProduct — media=%.2f | std=%.2f",
            df_feat["BalancePerProduct"].mean(),
            df_feat["BalancePerProduct"].std())

# PointsPerSalary: engajamento no programa de fidelidade vs renda
df_feat["PointsPerSalary"] = (
    df_feat["Point Earned"] / (df_feat["EstimatedSalary"] + 1)
)
logger.info("Feature criada: PointsPerSalary — media=%.6f | std=%.6f",
            df_feat["PointsPerSalary"].mean(),
            df_feat["PointsPerSalary"].std())

# --- 4. Separação de Features e Target + SPLIT (ANTES do scaling) ---

FEATURE_COLS = [c for c in df_feat.columns if c not in COLS_LEAKAGE]

X = df_feat[FEATURE_COLS].copy()
y = df_feat[TARGET_COL].copy()

# Split ANTES do scaling → evita leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

logger.info("Split treino/teste realizado ANTES do scaling (sem leakage) ✅")
logger.info("Shape X_train: %s | X_test: %s", X_train.shape, X_test.shape)
logger.info("Churn treino: %.2f%% | Churn teste: %.2f%%",
            y_train.mean()*100, y_test.mean()*100)

# --- 5. Scaling (fit apenas no treino) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Converter para DataFrame (facilita logs e análise)
X_train_final = pd.DataFrame(X_train_scaled, columns=FEATURE_COLS, index=X_train.index)
X_test_final  = pd.DataFrame(X_test_scaled,  columns=FEATURE_COLS, index=X_test.index)

logger.info("StandardScaler aplicado corretamente (fit only on train)")
logger.info("Features finais (%d): %s", len(FEATURE_COLS), FEATURE_COLS)

In [ ]:
df_feat.head()

## 🎯 Métricas de Avaliação

Problema de **churn** é desbalanceado (tipicamente ~20% de churn). Métricas comuns como Accuracy enganam bastante.

| Métrica      | O que mede                                      | Importância no Churn Bancário                          | Prioridade |
|--------------|--------------------------------------------------|---------------------------------------------------------|------------|
| **AUC-ROC**  | Capacidade de **ranquear** churners vs não-churners | Menos sensível ao threshold e ao desbalanceamento      | ★★★★★     |
| **Recall**   | % dos clientes que **realmente saíram** que foram identificados | Métrica de negócio #1. Perder um cliente custa caro (aquisição > retenção) | ★★★★★     |
| **Precision**| % dos clientes que o modelo **previu como churn** que realmente saíram | Evita acionar campanhas de retenção desnecessárias (custo operacional) | ★★★★      |
| **F1-Score** | Média harmônica Precision × Recall               | Bom equilíbrio quando as duas importam                  | ★★★★      |
| **Accuracy** | % de acertos totais                              | Quase inútil aqui. Um modelo "burro" que sempre diz "fica" já teria ~80% | ★         |

**Regra de negócio do Datathon**: delta AUC ≥ 0.005 para promover um modelo. Isso é conservador e saudável (evita promover ruído estatístico).

In [ ]:
# ============================================================
# Métricas de avaliação
# ============================================================
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
)

In [ ]:
# ============================================================
# Experimento 1: Baseline — Random Forest
# ============================================================

from sklearn.ensemble import RandomForestClassifier

MODEL_BASELINE_PARAMS = {
    "n_estimators":  100,
    "max_depth":     10,
    "random_state":  SEED,
    "class_weight":  "balanced",  # dataset desbalanceado — churn é minoria
}


logger.info("Treinando modelo Baseline Random Forest — %d amostras, %d features...",
            *X_train.shape)
logger.info("Parâmetros: %s", MODEL_BASELINE_PARAMS)
model = RandomForestClassifier(**MODEL_BASELINE_PARAMS)
model.fit(X_train_final, y_train)

# Avaliação
y_pred = model.predict(X_test_final)
y_pred_proba = model.predict_proba(X_test_final)[:, 1]

metrics = {
    "auc": roc_auc_score(y_test, y_pred_proba),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "accuracy": accuracy_score(y_test, y_pred),
}

logger.info("✅ Baseline treinado!")
for k, v in metrics.items():
    logger.info(f"   {k.upper():<9} = {v:.4f}")


In [ ]:
# ============================================================
# Segundo experimento: Challenger (Random Forest)
# ============================================================

from sklearn.ensemble import RandomForestClassifier

# Parâmetros do modelo challenger (mais robusto)
MODEL_CHALLENGER_PARAMS = {
    "n_estimators":     200,   # mais árvores
    "max_depth":        15,    # árvores mais profundas
    "random_state":     SEED,
    "class_weight":     "balanced",
    "min_samples_leaf": 5,     # ajuda a evitar overfitting
}

# --- Treinamento do Challenger ---
logger.info("Treinando Challenger Random Forest — %d amostras, %d features...",
            *X_train.shape)
logger.info("Parâmetros: %s", MODEL_CHALLENGER_PARAMS)
model_v2 = RandomForestClassifier(**MODEL_CHALLENGER_PARAMS)
model_v2.fit(X_train_final, y_train)

# --- Avaliação no conjunto de teste ---
y_pred_v2 = model_v2.predict(X_test_final)
y_pred_proba_v2 = model_v2.predict_proba(X_test_final)[:, 1]

metrics_v2 = {
    "auc": roc_auc_score(y_test, y_pred_proba_v2),
    "f1": f1_score(y_test, y_pred_v2, zero_division=0),
    "precision": precision_score(y_test, y_pred_v2, zero_division=0),
    "recall": recall_score(y_test, y_pred_v2, zero_division=0),
    "accuracy": accuracy_score(y_test, y_pred_v2),
}

logger.info("✅ Challenger RF treinado!")
for k, v in metrics_v2.items():
    logger.info(f"   {k.upper():<9} = {v:.4f}")

In [ ]:
# ============================================================
# --- Comparação direta: Baseline vs Challenger 1 ---
# Champion vs Challenger — comparação direta entre modelos
# ============================================================
print("\n" + "="*70)
print(f"{'Métrica':<12} {'Baseline':>12} {'Challenger RF':>15} {'Delta':>10}")
print("="*70)

for m in ["auc", "f1", "precision", "recall", "accuracy"]:
    delta = metrics_v2[m] - metrics[m]
    sinal = "↑" if delta > 0 else "↓" if delta < 0 else " "
    print(f"{m:<12} {metrics[m]:>12.4f} {metrics_v2[m]:>15.4f} {sinal}{abs(delta):>8.4f}")

print("="*70)


In [ ]:
# ============================================================
# Experimento 3: XGBoost — Geralmente o melhor em dados tabulares de churn
# ============================================================

import xgboost as xgb

xgb_params = {
    "n_estimators": 300,
    "max_depth": 8,
    "learning_rate": 0.05,
    "random_state": SEED,
    "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),  # corrige desbalanceamento
    "eval_metric": "auc",
    "verbosity": 0,
}

logger.info("Treinando XGBoost...")
model_xgb = xgb.XGBClassifier(**xgb_params)
model_xgb.fit(X_train_final, y_train)

y_pred_xgb = model_xgb.predict(X_test_final)
y_pred_proba_xgb = model_xgb.predict_proba(X_test_final)[:, 1]

metrics_xgb = {
    "auc": roc_auc_score(y_test, y_pred_proba_xgb),
    "f1": f1_score(y_test, y_pred_xgb, zero_division=0),
    "precision": precision_score(y_test, y_pred_xgb, zero_division=0),
    "recall": recall_score(y_test, y_pred_xgb, zero_division=0),
    "accuracy": accuracy_score(y_test, y_pred_xgb),
}

logger.info("✅ XGBoost treinado com sucesso!")
for k, v in metrics_xgb.items():
    logger.info(f"   {k.upper():<9} = {v:.4f}")

In [ ]:
# ============================================================
# Comparação Final: Baseline × Challenger RF × XGBoost
# ============================================================

import pandas as pd

# ====================== Dados originais ======================
data = {
    "Métrica": ["AUC", "F1-Score", "Precision", "Recall", "Accuracy"],
    "Baseline RF": [
        metrics["auc"], metrics["f1"], metrics["precision"],
        metrics["recall"], metrics["accuracy"]
    ],
    "Challenger RF": [
        metrics_v2["auc"], metrics_v2["f1"], metrics_v2["precision"],
        metrics_v2["recall"], metrics_v2["accuracy"]
    ],
    "XGBoost": [
        metrics_xgb["auc"], metrics_xgb["f1"], metrics_xgb["precision"],
        metrics_xgb["recall"], metrics_xgb["accuracy"]
    ]
}

df_comp = pd.DataFrame(data)

# ====================== Funções auxiliares ======================
def get_winner(row):
    vals = {
        "Baseline RF": row["Baseline RF"],
        "Challenger RF": row["Challenger RF"],
        "XGBoost": row["XGBoost"]
    }
    best_val = max(vals.values())
    best_model = [k for k, v in vals.items() if v == best_val][0]
    return f"{best_val:.4f} ({best_model})"

def add_highlight(row):
    best_val = max(row["Baseline RF"], row["Challenger RF"], row["XGBoost"])
    result = []
    for col in ["Baseline RF", "Challenger RF", "XGBoost"]:
        val = row[col]
        if abs(val - best_val) < 1e-6:
            result.append(f"{val:.4f} ↑")
        else:
            result.append(f"{val:.4f}")
    return result

# ====================== Aplicação ======================
df_comp["Melhor"] = df_comp.apply(get_winner, axis=1)

highlighted_cols = df_comp.apply(add_highlight, axis=1, result_type='expand')
highlighted_cols.columns = ["Baseline RF", "Challenger RF", "XGBoost"]

for col in ["Baseline RF", "Challenger RF", "XGBoost"]:
    df_comp[col] = highlighted_cols[col]

# ====================== Exibição Final Melhorada ======================
print("\n" + "="*105)
print(" " * 35 + "COMPARAÇÃO FINAL DOS MODELOS")
print("="*105)
print(df_comp.to_string(index=False, justify="center", col_space=12))
print("="*105)

# Resumo
print(f"\n🏆 Melhor modelo geral (maioria das métricas): Challenger RF")

# Critério de promoção do Datathon
delta_xgb = metrics_xgb["auc"] - metrics["auc"]
logger.info("Delta AUC (XGBoost vs Baseline): %.4f", delta_xgb)

if delta_xgb >= 0.005:
    logger.info("✅ XGBoost atende ao critério de promoção")
    print("\n✅ XGBoost atende ao critério de promoção do Datathon (delta AUC >= 0.005)!")
else:
    logger.info("⚠️ XGBoost não atende ao threshold de promoção")
    print("\n⚠️  Nenhum modelo superou o threshold de promoção exigido (delta AUC >= 0.005)")

### 🔍 Interpretação das Features Mais Importantes (XGBoost - Gain)

O gráfico de **Feature Importance** (usando o critério "Gain") revela quais variáveis o modelo considerou mais relevantes para prever o churn dos clientes. Quanto maior o valor de Gain, maior a contribuição da feature nas decisões do modelo.

#### Top 5 Features Mais Importantes

| Posição | Feature              | Importância (%) | Interpretação de Negócio |
|---------|----------------------|------------------|---------------------------|
| 1º     | **NumOfProducts**    | 26.59%          | **Mais importante do modelo**. Clientes com poucos produtos bancários (ex: apenas conta) têm risco muito maior de churn. Estratégia de cross-selling pode ser muito eficaz. |
| 2º     | **IsActiveMember**   | 15.34%          | Clientes inativos saem muito mais do banco. Manter o engajamento (uso de app, transações, etc.) é uma das chaves para reduzir o churn. |
| 3º     | **Age**              | 10.31%          | A idade tem influência significativa. Geralmente clientes muito jovens ou mais velhos apresentam maior probabilidade de churn. |
| 4º     | **Geo_Germany**      | 7.60%           | Clientes da Alemanha possuem maior risco de churn em relação à França (categoria referência). Pode refletir diferenças de mercado ou concorrência no país. |
| 5º     | **Balance**          | 4.81%           | O saldo em conta influencia a decisão de permanência. Clientes com saldos baixos tendem a ser mais propensos a sair. |

In [ ]:
# ============================================================
# Análise de Feature Importance — XGBoost (Gain)
# ============================================================

import xgboost as xgb
import matplotlib.pyplot as plt

# Gráfico de importância
plt.figure(figsize=(12, 10))
xgb.plot_importance(model_xgb,
                    max_num_features=15,
                    importance_type="gain")

plt.title("Feature Importance — XGBoost (Gain)", fontsize=14, pad=20)
plt.xlabel("Gain (contribuição para redução de erro)")
plt.tight_layout()
plt.show()

# Top 5 features em formato de tabela (para log e apresentação)
importances = pd.Series(model_xgb.feature_importances_,
                        index=FEATURE_COLS).sort_values(ascending=False)

logger.info("=== TOP 5 FEATURES MAIS IMPORTANTES (XGBoost - Gain) ===")
logger.info(importances.head(5).to_string(float_format="%.4f"))

print("\n📊 Top 10 Features mais importantes:")
print(importances.head(10).to_string(float_format="%.4f"))